<a href="https://colab.research.google.com/github/harishwork-s/aimlexercisesincollab/blob/main/creditcard_fraud_dectection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

Mounted at /content/drive


In [3]:
file_path = "/content/drive/MyDrive/aiml_project/paysim.csv"

dtypes = {
    "step": "int32",
    "type": "category",
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8"
}

cols = [
    "step","type","amount",
    "oldbalanceOrg","newbalanceOrig",
    "oldbalanceDest","newbalanceDest",
    "isFraud"
]

df = pd.read_csv(file_path, usecols=cols, dtype=dtypes)

print(df.shape)
df.head()

(6362620, 8)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.639648,170136.0,160296.359375,0.0,0.0,0
1,1,PAYMENT,1864.280029,21249.0,19384.720703,0.0,0.0,0
2,1,TRANSFER,181.000000,181.0,0.000000,0.0,0.0,1
3,1,CASH_OUT,181.000000,181.0,0.000000,21182.0,0.0,1
4,1,PAYMENT,11668.139648,41554.0,29885.859375,0.0,0.0,0


In [4]:
# Convert type to numeric
df["type"] = df["type"].map({
    "PAYMENT": 0,
    "TRANSFER": 1,
    "CASH_OUT": 2,
    "DEBIT": 3,
    "CASH_IN": 4
})

# Remove duplicates
df = df.drop_duplicates()

# Sort by time (VERY IMPORTANT)
df = df.sort_values("step").reset_index(drop=True)

print("Cleaned shape:", df.shape)

Cleaned shape: (6362077, 8)


In [5]:
print(df["isFraud"].value_counts())
print(df["isFraud"].value_counts(normalize=True))

isFraud
0    6353880
1       8197
Name: count, dtype: int64
isFraud
0    0.998712
1    0.001288
Name: proportion, dtype: float64


In [6]:
# Balance error
df["balance_error"] = df["oldbalanceOrg"] - df["newbalanceOrig"] - df["amount"]

# Destination balance error
df["dest_balance_error"] = df["newbalanceDest"] - df["oldbalanceDest"] - df["amount"]

# Zero balance flags
df["is_zero_orig"] = (df["oldbalanceOrg"] == 0).astype(int)
df["is_zero_dest"] = (df["oldbalanceDest"] == 0).astype(int)

# High risk transaction type
df["is_high_risk_type"] = df["type"].apply(lambda x: 1 if x in [1, 2] else 0)

# Log transform
df["amount_log"] = np.log1p(df["amount"])

# Relative amount
df["amount_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)

# Time feature
df["hour"] = df["step"] % 24

print("Features created")

Features created


In [7]:
df = df.drop(["amount"], axis=1)

df.head()

,step,type,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_error,dest_balance_error,is_zero_orig,is_zero_dest,is_high_risk_type,amount_log,amount_ratio,hour
0,1,1,10835.0,0.000000,6267.0,2.719173e+06,0,-300850.875000,2.401220e+06,0,0,1,12.649755,28.763924,1
1,1,0,67852.0,63975.589844,0.0,0.000000e+00,0,0.000244,-3.876410e+03,0,1,0,8.262922,0.057130,1
2,1,3,1817.0,751.590027,10330.0,0.000000e+00,0,-0.000122,-1.139541e+04,0,0,0,6.972053,0.586034,1
3,1,3,11299.0,1996.209961,29832.0,1.689670e+04,0,0.000000,-2.223809e+04,0,0,0,9.138177,0.823256,1
4,1,0,13854.0,12480.570312,0.0,0.000000e+00,0,-0.000366,-1.373430e+03,0,1,0,7.225794,0.099129,1


In [8]:
split_step = df["step"].quantile(0.8)

train_df = df[df["step"] <= split_step]
test_df = df[df["step"] > split_step]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (5113421, 15)
Test: (1248656, 15)


In [9]:
X_train = train_df.drop("isFraud", axis=1)
y_train = train_df["isFraud"]

X_test = test_df.drop("isFraud", axis=1)
y_test = test_df["isFraud"]

print(X_train.shape, X_test.shape)

(5113421, 14) (1248656, 14)


In [10]:
print(train_df["step"].max())
print(test_df["step"].min())

355
356


In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [11]:
# Calculate class weight manually
fraud = sum(y_train == 1)
normal = sum(y_train == 0)

weight_for_1 = normal / fraud

class_weights = {0: 1, 1: weight_for_1}

print("Class Weights:", class_weights)

Class Weights: {0: 1, 1: 1289.9419338550872}


In [12]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    class_weight=class_weights,
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight={0: 1, 1: 1289.9419338550872}, max_depth=10,
                       n_jobs=-1, random_state=42)

In [13]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

In [14]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[1244420       0]
 [      2    4234]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1244420
           1       1.00      1.00      1.00      4236

    accuracy                           1.00   1248656
   macro avg       1.00      1.00      1.00   1248656
weighted avg       1.00      1.00      1.00   1248656


ROC-AUC Score:
0.9999999901353788


In [15]:
print(X_train.columns)

Index(['step', 'type', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
       'newbalanceDest', 'balance_error', 'dest_balance_error', 'is_zero_orig',
       'is_zero_dest', 'is_high_risk_type', 'amount_log', 'amount_ratio',
       'hour'],
      dtype='object')


In [16]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print(feature_importance.head(10))

               feature  importance
6        balance_error    0.311051
12        amount_ratio    0.241113
10   is_high_risk_type    0.129501
3       newbalanceOrig    0.100773
2        oldbalanceOrg    0.075728
1                 type    0.041562
11          amount_log    0.030858
13                hour    0.021513
7   dest_balance_error    0.013500
8         is_zero_orig    0.010772


In [17]:
X_train_hard = X_train.drop(["balance_error", "amount_ratio"], axis=1)
X_test_hard = X_test.drop(["balance_error", "amount_ratio"], axis=1)

In [18]:
rf_model_hard = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    class_weight=class_weights,
    random_state=42
)

rf_model_hard.fit(X_train_hard, y_train)

RandomForestClassifier(class_weight={0: 1, 1: 1289.9419338550872}, max_depth=10,
                       n_jobs=-1, random_state=42)

In [19]:
y_pred_hard = rf_model_hard.predict(X_test_hard)
y_prob_hard = rf_model_hard.predict_proba(X_test_hard)[:, 1]

print(confusion_matrix(y_test, y_pred_hard))
print(classification_report(y_test, y_pred_hard))
print(roc_auc_score(y_test, y_prob_hard))

[[1227227   17193]
 [     56    4180]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99   1244420
           1       0.20      0.99      0.33      4236

    accuracy                           0.99   1248656
   macro avg       0.60      0.99      0.66   1248656
weighted avg       1.00      0.99      0.99   1248656

0.999059572185192


In [20]:
threshold = 0.90  # change this

y_pred_custom = (y_prob_hard > threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[1244390      30]
 [   1005    3231]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1244420
           1       0.99      0.76      0.86      4236

    accuracy                           1.00   1248656
   macro avg       0.99      0.88      0.93   1248656
weighted avg       1.00      1.00      1.00   1248656



In [21]:
from sklearn.metrics import precision_score, recall_score

for t in [0.5, 0.7, 0.8, 0.9, 0.95]:
    y_pred_t = (y_prob_hard > t).astype(int)
    print(f"\nThreshold: {t}")
    print("Precision:", precision_score(y_test, y_pred_t))
    print("Recall:", recall_score(y_test, y_pred_t))


Threshold: 0.5
Precision: 0.19557385486361298
Recall: 0.9867799811142587

Threshold: 0.7
Precision: 0.429525791260541
Recall: 0.9258734655335222

Threshold: 0.8
Precision: 0.9103879849812265
Recall: 0.8585930122757318

Threshold: 0.9
Precision: 0.9908003679852806
Recall: 0.7627478753541076

Threshold: 0.95
Precision: 0.9972789115646259
Recall: 0.5191218130311614


In [22]:
threshold = 0.8
y_pred_final = (y_prob_hard > threshold).astype(int)

In [23]:
def risk_level(prob):
    if prob > 0.9:
        return "HIGH"
    elif prob > 0.8:
        return "MEDIUM"
    else:
        return "LOW"

In [24]:
results = pd.DataFrame({
    "Probability": y_prob_hard[:10],
    "Prediction": y_pred_final[:10]
})

results["Risk"] = results["Probability"].apply(risk_level)

results

,Probability,Prediction,Risk
0,0.073973,0,LOW
1,0.000000,0,LOW
2,0.000000,0,LOW
3,0.000000,0,LOW
4,0.052721,0,LOW
5,0.000000,0,LOW
6,0.000000,0,LOW
7,0.000000,0,LOW
8,0.000000,0,LOW
9,0.000000,0,LOW


In [25]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.2 MB/s eta 0:00:00


In [26]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [28]:
X_train_hard["type"] = X_train_hard["type"].astype(int)
X_test_hard["type"] = X_test_hard["type"].astype(int)

In [29]:
cat_model.fit(X_train_hard, y_train)

CatBoostClassifier(depth=6, eval_metric='AUC', iterations=200, learning_rate=0.1, loss_function='Logloss', verbose=0)

In [30]:
y_pred_cat = cat_model.predict(X_test_hard)
y_prob_cat = cat_model.predict_proba(X_test_hard)[:, 1]

In [31]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_cat))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob_cat))

Confusion Matrix:
[[1244082     338]
 [    724    3512]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1244420
           1       0.91      0.83      0.87      4236

    accuracy                           1.00   1248656
   macro avg       0.96      0.91      0.93   1248656
weighted avg       1.00      1.00      1.00   1248656


ROC-AUC Score:
0.9995110905013882


In [32]:
from sklearn.metrics import precision_score, recall_score

for t in [0.5, 0.7, 0.8, 0.9]:
    y_pred_t = (y_prob_cat > t).astype(int)
    print(f"\nThreshold: {t}")
    print("Precision:", precision_score(y_test, y_pred_t))
    print("Recall:", recall_score(y_test, y_pred_t))


Threshold: 0.5
Precision: 0.9122077922077922
Recall: 0.8290840415486308

Threshold: 0.7
Precision: 0.942578548212351
Recall: 0.8215297450424929

Threshold: 0.8
Precision: 0.9524861878453039
Recall: 0.8139754485363551

Threshold: 0.9
Precision: 0.9681274900398407
Recall: 0.8031161473087819


In [33]:
import joblib

joblib.dump(cat_model, "/content/drive/MyDrive/aiml_project/catboost_model.pkl")

['/content/drive/MyDrive/aiml_project/catboost_model.pkl']

In [34]:
pip install fastapi uvicorn pandas joblib

In [36]:
import os

print(os.listdir("/content/drive/MyDrive/aiml_project"))

['paysim.csv', 'paysim_clean.csv', 'catboost_model.pkl']


In [37]:
from fastapi import FastAPI, UploadFile, File
import pandas as pd
import joblib
import numpy as np

app = FastAPI()

# Load model
model = joblib.load("/content/drive/MyDrive/aiml_project/catboost_model.pkl")

# Same preprocessing function (VERY IMPORTANT)
def preprocess(df):
    df["type"] = df["type"].map({
        "PAYMENT": 0,
        "TRANSFER": 1,
        "CASH_OUT": 2,
        "DEBIT": 3,
        "CASH_IN": 4
    })

    df["balance_error"] = df["oldbalanceOrg"] - df["newbalanceOrig"] - df["amount"]
    df["dest_balance_error"] = df["newbalanceDest"] - df["oldbalanceDest"] - df["amount"]
    df["is_zero_orig"] = (df["oldbalanceOrg"] == 0).astype(int)
    df["is_zero_dest"] = (df["oldbalanceDest"] == 0).astype(int)
    df["is_high_risk_type"] = df["type"].apply(lambda x: 1 if x in [1, 2] else 0)
    df["amount_log"] = np.log1p(df["amount"])
    df["amount_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)
    df["hour"] = df["step"] % 24

    df = df.drop(["amount"], axis=1)

    # Remove strong features (same as training)
    df = df.drop(["balance_error", "amount_ratio"], axis=1)

    return df

def risk_level(prob):
    if prob > 0.9:
        return "HIGH"
    elif prob > 0.8:
        return "MEDIUM"
    else:
        return "LOW"

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    df = pd.read_csv(file.file)

    df_processed = preprocess(df)

    probs = model.predict_proba(df_processed)[:, 1]

    results = []
    for i in range(len(probs)):
        results.append({
            "probability": float(probs[i]),
            "risk": risk_level(probs[i]),
            "prediction": "FRAUD" if probs[i] > 0.8 else "NORMAL"
        })

    return {"results": results}